In [2]:

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


/opt/anaconda3/envs/ai/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv("../data/features/features_complete.csv")

# Optional: check columns
print(df.columns)


Index(['index', 'video_id', 'trending_date', 'title', 'channel_title',
       'category_id', 'publish_date', 'time_frame', 'published_day_of_week',
       'publish_country', 'tags', 'views', 'likes', 'dislikes',
       'comment_count', 'comments_disabled', 'ratings_disabled',
       'video_error_or_removed', 'category_name', 'is_published_weekend',
       'trending_day_of_week', 'is_trending_weekend', 'hour_of_trending',
       'days_until_trending', 'title_cleaned', 'title_length',
       'uppercase_words', 'contains_numbers_or_emojis', 'num_emojis',
       'has_emoji', 'emojis_list', 'sentiment_polarity',
       'sentiment_subjectivity', 'is_title_english', 'top50_pca1',
       'top50_pca2', 'top50_pca3', 'published_day_of_week_num'],
      dtype='object')


In [4]:
# Use the features you want for tabular input
tabular_cols = [
    'title_length', 
    'uppercase_words', 
    'sentiment_polarity', 
    'sentiment_subjectivity',
    'category_id',  
    'published_day_of_week_num',
    'hour_of_trending',       # Time of day
    'days_until_trending',    # How long to trend
    'num_emojis',
    'has_emoji',
    'contains_numbers_or_emojis',
    'comments_disabled',
    'ratings_disabled',
    'is_title_english',             # Title language flag

    'top50_pca1',             # PCA embedding of top 50 words
    'top50_pca2',
    'top50_pca3',

    # --- Metadata / flags ---
    'is_published_weekend',   # True if video published on weekend
    'is_trending_weekend',    # True if trending on weekend
]


In [5]:
# -----------------------------
# SPLIT DATA
# -----------------------------
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
X_train_tab = train_df[tabular_cols].astype(float)
X_test_tab = test_df[tabular_cols].astype(float)
y_train = np.log1p(train_df['views']).values
y_test = np.log1p(test_df['views']).values

In [6]:
# -----------------------------
# TEXT EMBEDDINGS
# -----------------------------
train_titles = train_df['title_cleaned'].tolist()
test_titles = test_df['title_cleaned'].tolist()
text_model = SentenceTransformer("all-MiniLM-L6-v2")
X_train_text = text_model.encode(train_titles, show_progress_bar=True)
X_test_text = text_model.encode(test_titles, show_progress_bar=True)

'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x32e91b7d0>: Failed to resolve \'huggingface.co\' ([Errno 8] nodename nor servname provided, or not known)"))'), '(Request ID: b1665c9c-6927-476f-a8f7-0a2bcf5bba17)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x32e9323d0>: Failed to resolve \'huggingface.co\' ([Errno 8] nodename nor servname provided, or not known)"))'), '(Request ID: bd1c6236-6d27-4f84-b259-509fb800859c)')' thrown while reques

In [7]:
class MultimodalDataset(Dataset):
    def __init__(self, X_tab, X_text, y):
        self.X_tab = torch.tensor(X_tab, dtype=torch.float32)
        self.X_text = torch.tensor(X_text, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_tab[idx], self.X_text[idx], self.y[idx]

train_ds = MultimodalDataset(X_train_tab.values, X_train_text, y_train)
test_ds = MultimodalDataset(X_test_tab.values, X_test_text, y_test)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64)

In [8]:
class MultimodalModel(nn.Module):
    def __init__(self, tabular_input_dim, text_input_dim, hidden_dim=128):
        super().__init__()
        self.tab_fc = nn.Sequential(
            nn.Linear(tabular_input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim//2),
            nn.ReLU()
        )
        self.text_fc = nn.Sequential(
            nn.Linear(text_input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim//2),
            nn.ReLU()
        )
        self.combined_fc = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim//2),
            nn.ReLU(),
            nn.Linear(hidden_dim//2, 1)
        )

    def forward(self, x_tab, x_text):
        tab_out = self.tab_fc(x_tab)
        text_out = self.text_fc(x_text)
        combined = torch.cat([tab_out, text_out], dim=1)
        return self.combined_fc(combined).squeeze()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultimodalModel(tabular_input_dim=X_train_tab.shape[1], text_input_dim=X_train_text.shape[1])
model.to(device)

MultimodalModel(
  (tab_fc): Sequential(
    (0): Linear(in_features=19, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
  )
  (text_fc): Sequential(
    (0): Linear(in_features=384, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
  )
  (combined_fc): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=1, bias=True)
  )
)

In [9]:
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64)

In [10]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()
n_epochs = 10

for epoch in range(n_epochs):
    model.train()
    train_losses = []
    for X_tab_batch, X_text_batch, y_batch in train_loader:
        X_tab_batch, X_text_batch, y_batch = X_tab_batch.to(device), X_text_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        y_pred = model(X_tab_batch, X_text_batch)
        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
    print(f"Epoch {epoch+1}/{n_epochs} | Train Loss: {np.mean(train_losses):.4f}")

Epoch 1/10 | Train Loss: 4.9792
Epoch 2/10 | Train Loss: 1.7702
Epoch 3/10 | Train Loss: 1.6316
Epoch 4/10 | Train Loss: 1.5464
Epoch 5/10 | Train Loss: 1.4712
Epoch 6/10 | Train Loss: 1.4152
Epoch 7/10 | Train Loss: 1.3666
Epoch 8/10 | Train Loss: 1.3275
Epoch 9/10 | Train Loss: 1.2970
Epoch 10/10 | Train Loss: 1.2546


In [ ]:
model.eval()
y_preds = []
y_trues = []
with torch.no_grad():
    for X_tab_batch, X_text_batch, y_batch in test_loader:
        X_tab_batch, X_text_batch = X_tab_batch.to(device), X_text_batch.to(device)
        y_pred = model(X_tab_batch, X_text_batch)
        y_preds.extend(y_pred.cpu().numpy())
        y_trues.extend(y_batch.numpy())

y_preds = np.array(y_preds)
y_trues = np.array(y_trues)
rmse = np.sqrt(mean_squared_error(y_trues, y_preds))
mae = mean_absolute_error(y_trues, y_preds)
r2 = r2_score(y_trues, y_preds)


print(f"\nTest RMSE: {rmse:.4f} | MAE: {mae:.4f} | R²: {r2:.4f}")

[10.762615 10.841422 10.280793 ... 11.431553 12.718097  8.903543] tensor([12.3006,  8.5282, 10.0913, 12.3944, 12.7681, 11.0987, 12.4944, 13.1444,
         9.7268,  9.0158, 12.4796, 10.2738,  9.5324, 11.7521,  9.4052, 12.0196,
        12.1654, 10.4785, 12.7715, 11.0373, 10.5904,  9.9104, 13.0500, 11.7049,
        13.1238,  8.7227, 10.2571, 10.0187,  8.7787, 11.1427, 10.0259, 12.8726,
         9.5443, 11.8177, 10.6075, 10.9041, 11.5615,  9.3889, 13.1565, 11.7428,
        13.3616, 10.1617])

Test RMSE: 1.2536 | MAE: 0.9721 | R²: 0.4420


In [12]:
# Save PyTorch model
torch.save({
    'model_state_dict': model.state_dict(),
    'model_config': {
        'tabular_input_dim': X_train_tab.shape[1],
        'text_embedding_dim': X_train_text.shape[1],
        'hidden_dims': [128, 64],
        'dropout': 0.3
    },
    'metrics': {'rmse': rmse, 'mae': mae, 'r2': r2}
}, '../models/multimodal_pytorch.pth')


print("✅ Model, scaler, and PCA saved successfully.")

✅ Model, scaler, and PCA saved successfully.


In [13]:
# ========================================
# ABLATION STUDY
# ========================================

print("\n" + "="*70)
print("ABLATION STUDY: Understanding Component Contributions")
print("="*70)

ablation_results = []

# ----------------------------------------
# Experiment 1: Metadata Only (No Text)
# ----------------------------------------
print("\n1️⃣ Training: Metadata Only Model...")

class MetadataOnlyModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )
    
    def forward(self, x_tabular, x_text=None):
        # Ignore text input
        return self.fc(x_tabular).squeeze()

# Train metadata-only model
metadata_model = MetadataOnlyModel(input_dim=X_train_tab.shape[1]).to(device)
# ... train with same training loop ...

# Evaluate
metadata_model.eval()
y_preds_meta = []
with torch.no_grad():
    for X_tab_batch, X_text_batch, y_batch in test_loader:
        X_tab_batch = X_tab_batch.to(device)
        y_pred = metadata_model(X_tab_batch)
        y_preds_meta.extend(y_pred.cpu().numpy())

y_preds_meta = np.array(y_preds_meta)
rmse_meta = np.sqrt(mean_squared_error(y_trues, y_preds_meta))
r2_meta = r2_score(y_trues, y_preds_meta)

ablation_results.append({
    'Model': 'Metadata Only',
    'RMSE': rmse_meta,
    'MAE': mean_absolute_error(y_trues, y_preds_meta),
    'R²': r2_meta,
    'Components': 'Tabular features'
})

print(f"   RMSE: {rmse_meta:.4f}, R²: {r2_meta:.4f}")

# ----------------------------------------
# Experiment 2: Full Multimodal (Your Current Model)
# ----------------------------------------
ablation_results.append({
    'Model': 'Multimodal (Tabular + Text)',
    'RMSE': rmse,
    'MAE': mae,
    'R²': r2,
    'Components': 'Tabular + Text embeddings'
})

print(f"\n2️⃣ Multimodal Model (already trained):")
print(f"   RMSE: {rmse:.4f}, R²: {r2:.4f}")

# ----------------------------------------
# Display Ablation Results
# ----------------------------------------
print("\n" + "="*70)
print("ABLATION STUDY RESULTS")
print("="*70)

ablation_df = pd.DataFrame(ablation_results)
print(ablation_df.to_string(index=False))

print("\n📊 Key Insights:")
improvement = ((r2 - r2_meta) / r2_meta) * 100
print(f"   • Adding text embeddings improves R² by {improvement:.1f}%")
print(f"   • Text contributes {r2 - r2_meta:.4f} additional variance explained")


ABLATION STUDY: Understanding Component Contributions

1️⃣ Training: Metadata Only Model...
   RMSE: 10.2453, R²: -36.2715

2️⃣ Multimodal Model (already trained):
   RMSE: 1.2536, R²: 0.4420

ABLATION STUDY RESULTS
                      Model      RMSE       MAE         R²                Components
              Metadata Only 10.245330 10.055269 -36.271477          Tabular features
Multimodal (Tabular + Text)  1.253591  0.972069   0.441997 Tabular + Text embeddings

📊 Key Insights:
   • Adding text embeddings improves R² by -101.2%
   • Text contributes 36.7135 additional variance explained


In [15]:
y_pred_new = model(X_tab_batch,X_text_batch)
print(y_pred_new)

tensor([12.3006,  8.5282, 10.0913, 12.3944, 12.7681, 11.0987, 12.4944, 13.1444,
         9.7268,  9.0158, 12.4796, 10.2738,  9.5324, 11.7521,  9.4052, 12.0196,
        12.1654, 10.4785, 12.7715, 11.0373, 10.5904,  9.9104, 13.0500, 11.7049,
        13.1238,  8.7227, 10.2571, 10.0187,  8.7787, 11.1427, 10.0259, 12.8726,
         9.5443, 11.8177, 10.6075, 10.9041, 11.5615,  9.3889, 13.1565, 11.7428,
        13.3616, 10.1617], grad_fn=<SqueezeBackward0>)
